## Imports

In [15]:
import os, re, json, base64, textwrap
from pathlib import Path

import cv2
import fitz            # PyMuPDF
import numpy as np
from portkey_ai import Portkey
from doclayout_yolo import YOLOv10

## Configuration

In [26]:
# ── Edit these values ──────────────────────────────────────────────────────
PDF_PATH        = "..\\..\\Sample PDFs\\OP-RAG.pdf"
YOLO_WEIGHTS    = "..\\..\\yolo\\doclayout_yolo_docstructbench_imgsz1024.pt"
PORTKEY_API_KEY = os.environ.get("PORTKEY_API_KEY")
PORTKEY_BASE    = os.environ.get("PORTKEY_BASE_URL")
MODEL_NAME      = "gpt-5.4-mini"
OUTPUT_DIR      = "output_OP_RAG"          # crops + final files go here

YOLO_IMG_SIZE   = 1024
YOLO_CONF       = 0.20
RENDER_ZOOM     = 2.0
JPEG_QUALITY    = 85

In [27]:
def setup():
    """Load PDF, create Portkey client, initialise YOLO model."""
    print("=" * 60)
    print("SETUP")
    print("=" * 60)

    # ── PDF ───────────────────────────────────────────────────────────────
    doc = fitz.open(PDF_PATH)
    print(f"  [IN]  PDF path   : {PDF_PATH}")
    print(f"  [OUT] Pages      : {len(doc)}")

    # ── Portkey client ────────────────────────────────────────────────────
    client = Portkey(base_url=PORTKEY_BASE, api_key=PORTKEY_API_KEY)
    print(f"  [OUT] Client     : Portkey  model={MODEL_NAME}")

    # ── YOLO ──────────────────────────────────────────────────────────────
    model = YOLOv10(YOLO_WEIGHTS)
    print(f"  [OUT] YOLO model : {YOLO_WEIGHTS}")

    # ── Output folders ────────────────────────────────────────────────────
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    print(f"  [OUT] Output dir : {OUTPUT_DIR}/")
    print("=" * 60)

    return doc, client, model


# Run setup
doc, client, model = setup()

SETUP
  [IN]  PDF path   : ..\..\Sample PDFs\OP-RAG.pdf
  [OUT] Pages      : 5
  [OUT] Client     : Portkey  model=gpt-5.4-mini
  [OUT] YOLO model : ..\..\yolo\doclayout_yolo_docstructbench_imgsz1024.pt
  [OUT] Output dir : output_OP_RAG/


## Image Helpers

In [28]:
def render_page(page: fitz.Page, zoom: float = RENDER_ZOOM) -> np.ndarray:
    """Render a PDF page → BGR numpy array."""
    mat    = fitz.Matrix(zoom, zoom)
    pix    = page.get_pixmap(matrix=mat, alpha=False)
    img    = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)
    conv   = cv2.COLOR_RGBA2BGR if pix.n == 4 else cv2.COLOR_RGB2BGR
    return cv2.cvtColor(img, conv)


def to_data_url(img: np.ndarray, quality: int = JPEG_QUALITY) -> str:
    """BGR numpy array → base64 JPEG data URL."""
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        raise RuntimeError("JPEG encode failed")
    return "data:image/jpeg;base64," + base64.b64encode(buf).decode()


def save_image(img: np.ndarray, path: str) -> None:
    """Save a BGR image to disk."""
    cv2.imwrite(path, img)

In [29]:
VISUAL_LABELS = {"figure", "table", "figure_caption", "table_caption",
                  "isolate_formula", "formula_caption"}

def get_yolo_outputs(page_img: np.ndarray, page_num: int) -> list[dict]:
    """
    Run YOLO on a page image.
    Prints every detected block (type, confidence, location).
    Saves every crop image to output/page_<N>/.
    """
    print(f"\n{'─'*60}")
    print(f"  STAGE 1 · YOLO DETECTION  —  Page {page_num}")
    print(f"{'─'*60}")
    print(f"  [IN]  page_img shape : {page_img.shape}  dtype={page_img.dtype}")

    # ── Run model ─────────────────────────────────────────────────────────
    result = model.predict(page_img, imgsz=YOLO_IMG_SIZE, conf=YOLO_CONF, device="cpu")[0]
    boxes  = result.boxes
    names  = getattr(result, "names", {})
    H, W   = page_img.shape[:2]

    # ── Save folder for this page ─────────────────────────────────────────
    page_dir = Path(OUTPUT_DIR) / f"page_{page_num:03d}"
    page_dir.mkdir(parents=True, exist_ok=True)

    blocks = []
    if boxes is None or len(boxes) == 0:
        print("  [OUT] No blocks detected.")
        return blocks

    print(f"  [OUT] {len(boxes)} block(s) detected  (saved to {page_dir}/)\n")
    print(f"  {'IDX':>4}  {'LABEL':<20}  {'CONF':>5}  {'BBOX (x1,y1,x2,y2)'}")
    print(f"  {'─'*4}  {'─'*20}  {'─'*5}  {'─'*28}")

    for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy().astype(int).tolist()
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(x2, W),  min(y2, H)
        if x2 <= x1 or y2 <= y1:
            continue

        cls_id = int(boxes.cls[i].item())
        label  = names.get(cls_id, f"class_{cls_id}") if isinstance(names, dict) else str(cls_id)
        conf   = float(boxes.conf[i].item())
        crop   = page_img[y1:y2, x1:x2].copy()

        # Save crop
        crop_filename = f"block_{i+1:02d}_{label}.jpg"
        crop_path     = str(page_dir / crop_filename)
        save_image(crop, crop_path)

        block = {
            "idx"       : i + 1,
            "page"      : page_num,
            "label"     : label,
            "conf"      : round(conf, 3),
            "bbox"      : (x1, y1, x2, y2),
            "crop"      : crop,
            "crop_path" : crop_path,
        }
        blocks.append(block)

        # Mark visual blocks
        tag = "  ◆ VISUAL" if label in VISUAL_LABELS else ""
        print(f"  {i+1:>4}  {label:<20}  {conf:>5.2f}  ({x1:4d},{y1:4d},{x2:4d},{y2:4d}){tag}")

    # Sort top-to-bottom, left-to-right
    blocks.sort(key=lambda b: (b["bbox"][1], b["bbox"][0]))

    print(f"\n  [OUT] {len(blocks)} valid block(s)  |  crops → {page_dir}/")
    return blocks

##  `get_reading_order()`  — LLM Call 1

In [30]:
READING_ORDER_PROMPT = """
You are a document layout analyst.

You will receive:
  1. A full page image from a PDF
  2. A list of detected blocks with their bounding boxes and labels (from YOLO)

Your task: determine the correct READING ORDER of all blocks on this page.

Return ONLY a JSON array (no extra text). Each element:
{
  "order"     : <int>,        // 1-based reading position
  "idx"       : <int>,        // matches the block idx provided
  "label"     : <str>,        // block label (title / plain_text / figure / table / ...)
  "bbox"      : [x1, y1, x2, y2],
  "role"      : <str>,        // "heading" | "body" | "figure" | "table" |
                               // "caption" | "formula" | "skip"
  "note"      : <str>         // short note e.g. "section heading", "main paragraph",
                               // "figure with 2 subplots", "table header"
}

Rules:
- For multi-column layouts, read left column first then right column.
- Captions should appear immediately after the element they describe.
- Mark page numbers, headers, footers, watermarks as role="skip".
- Do NOT include any text outside the JSON array in your response.
"""


def get_reading_order(page_img: np.ndarray, blocks: list[dict], page_num: int) -> list[dict]:
    """
    LLM Call 1: send full page image + YOLO block list → reading order JSON.
    """
    print(f"\n{'─'*60}")
    print(f"  STAGE 2 · READING ORDER  (LLM Call 1)  —  Page {page_num}")
    print(f"{'─'*60}")

    # ── Build block summary for prompt ───────────────────────────────────
    block_summary = []
    for b in blocks:
        block_summary.append({
            "idx"   : b["idx"],
            "label" : b["label"],
            "conf"  : b["conf"],
            "bbox"  : list(b["bbox"]),
        })

    blocks_text = json.dumps(block_summary, indent=2)
    print(f"  [IN]  page_img shape : {page_img.shape}")
    print(f"  [IN]  blocks ({len(blocks)}) :")
    for b in block_summary:
        print(f"          idx={b['idx']:>2}  {b['label']:<20}  bbox={b['bbox']}")

    # ── LLM call ──────────────────────────────────────────────────────────
    content = [
        {"type": "input_text",  "text": READING_ORDER_PROMPT},
        {"type": "input_text",  "text": f"Page {page_num} · Detected blocks:\n{blocks_text}"},
        {"type": "input_text",  "text": "Full page image:"},
        {"type": "input_image", "image_url": to_data_url(page_img)},
    ]

    response = client.responses.create(
        model=MODEL_NAME,
        input=[{"role": "user", "content": content}],
    )
    raw = response.output_text.strip()

    # ── Parse JSON ────────────────────────────────────────────────────────
    try:
        # Strip possible markdown fences
        clean = re.sub(r"^```[a-z]*\n?|\n?```$", "", raw.strip(), flags=re.MULTILINE)
        reading_order = json.loads(clean)
    except json.JSONDecodeError as e:
        print(f"  [WARN] JSON parse error: {e}  — falling back to YOLO order")
        reading_order = [
            {"order": i+1, "idx": b["idx"], "label": b["label"],
             "bbox": list(b["bbox"]), "role": "body", "note": ""}
            for i, b in enumerate(blocks)
        ]

    # ── Print result ──────────────────────────────────────────────────────
    print(f"\n  [OUT] Reading order ({len(reading_order)} items) :")
    print(f"  {'ORD':>4}  {'IDX':>4}  {'LABEL':<20}  {'ROLE':<10}  NOTE")
    print(f"  {'─'*4}  {'─'*4}  {'─'*20}  {'─'*10}  {'─'*25}")
    for item in reading_order:
        if item.get("role") == "skip":
            continue
        print(f"  {item['order']:>4}  {item['idx']:>4}  {item['label']:<20}  "
              f"{item.get('role','')::<10}  {item.get('note','')[:40]}")

    # Save reading order JSON for this page
    ro_path = Path(OUTPUT_DIR) / f"page_{page_num:03d}" / "reading_order.json"
    ro_path.parent.mkdir(parents=True, exist_ok=True)
    with open(ro_path, "w") as f:
        json.dump(reading_order, f, indent=2)
    print(f"\n  [OUT] Saved → {ro_path}")

    return reading_order

## `generate_markdown()`  — LLM Call 2


In [31]:
MARKDOWN_PROMPT = """
You are converting a PDF page into clean Markdown.

You will receive:
  1. The full page image
  2. A READING ORDER JSON — use this as the strict sequence for your output
  3. Cropped images of each block (in the order given by the reading order)

=== MARKDOWN RULES ===
- Follow the reading order JSON exactly. Do NOT reorder content.
- Use #/##/### for headings (role=heading).
- Render tables as Markdown pipe tables.
- For figures/charts: write a placeholder  [Figure: <detailed description>]
- Skip all items with role="skip" (page numbers, headers, footers).
- Do NOT hallucinate. Only transcribe visible text.

=== JSON RULES ===
After the Markdown, append a fenced ```json block with an array of all
figures and tables on this page. Each element:
{
  "page"       : <int>,
  "type"       : "figure" | "table",
  "figure_tag" : "Figure N" | "Table N",   // N = local count on this page
  "heading"    : "<nearest heading above>",
  "caption"    : "<caption text or empty string>",
  "content"    : "<detailed description of what is shown>",
  "subplots"   : [{"label":"a","content":"..."}]   // only if multiple panels
}
If no figures or tables: ```json\n[]\n```
"""


def generate_markdown(
    page_img:      np.ndarray,
    blocks:        list[dict],
    reading_order: list[dict],
    page_num:      int,
) -> tuple[str, list[dict]]:
    """
    LLM Call 2: use reading order JSON + crops → Markdown + figures JSON.
    """
    print(f"\n{'─'*60}")
    print(f"  STAGE 3 · MARKDOWN GENERATION  (LLM Call 2)  —  Page {page_num}")
    print(f"{'─'*60}")
    print(f"  [IN]  page_img shape    : {page_img.shape}")
    print(f"  [IN]  reading_order len : {len(reading_order)} items")
    print(f"  [IN]  blocks (crops)    : {len(blocks)}")

    # ── Build block lookup by idx ─────────────────────────────────────────
    block_by_idx = {b["idx"]: b for b in blocks}

    # ── Assemble multimodal content ───────────────────────────────────────
    ro_clean = [{k: v for k, v in item.items()} for item in reading_order]
    content = [
        {"type": "input_text",  "text": MARKDOWN_PROMPT},
        {"type": "input_text",  "text": f"Page {page_num}"},
        {"type": "input_text",  "text": "Reading order JSON:\n" + json.dumps(ro_clean, indent=2)},
        {"type": "input_text",  "text": "Full page image:"},
        {"type": "input_image", "image_url": to_data_url(page_img)},
        {"type": "input_text",  "text": "Cropped blocks in reading order:"},
    ]

    included = 0
    for item in reading_order:
        if item.get("role") == "skip":
            continue
        blk = block_by_idx.get(item["idx"])
        if blk is None:
            continue
        content.append({
            "type": "input_text",
            "text": (f"Block idx={item['idx']}  label={item['label']}  "
                     f"role={item.get('role','')}  order={item['order']}"),
        })
        content.append({"type": "input_image", "image_url": to_data_url(blk["crop"])})
        included += 1

    print(f"  [IN]  crops sent to LLM : {included}")

    # ── LLM call ──────────────────────────────────────────────────────────
    response = client.responses.create(
        model=MODEL_NAME,
        input=[{"role": "user", "content": content}],
    )
    raw = response.output_text.strip()

    # ── Split Markdown from JSON block ────────────────────────────────────
    json_fence = re.compile(r"```json\s*(\[.*?\])\s*```", re.DOTALL)
    match      = json_fence.search(raw)

    figures_json: list[dict] = []
    if match:
        try:
            figures_json = json.loads(match.group(1))
            for elem in figures_json:
                elem["page"] = page_num
        except json.JSONDecodeError as e:
            print(f"  [WARN] figures JSON parse error: {e}")
        markdown = raw[: match.start()].strip()
    else:
        markdown = raw

    # ── Print outputs ─────────────────────────────────────────────────────
    preview = textwrap.shorten(markdown.replace("\n", " "), width=200)
    print(f"\n  [OUT] Markdown preview  : {preview}")
    print(f"  [OUT] Markdown length   : {len(markdown)} chars")
    print(f"  [OUT] Figures/Tables    : {len(figures_json)} element(s)")
    for elem in figures_json:
        print(f"          {elem.get('figure_tag','?'):>10}  type={elem.get('type','?'):<8}  "
              f"caption={elem.get('caption','')[:50]}")

    # Save page markdown + figures JSON
    pg_dir = Path(OUTPUT_DIR) / f"page_{page_num:03d}"
    with open(pg_dir / "markdown.md", "w", encoding="utf-8") as f:
        f.write(markdown)
    with open(pg_dir / "figures.json", "w", encoding="utf-8") as f:
        json.dump(figures_json, f, indent=2)
    print(f"  [OUT] Saved → {pg_dir}/markdown.md")
    print(f"  [OUT] Saved → {pg_dir}/figures.json")

    return markdown, figures_json

## `resolve_references()`


In [32]:
def resolve_references(
    full_markdown: str,
    all_figures:   list[dict],
) -> tuple[str, dict]:
    """
    1. Assign global Figure / Table numbers (sorted by page).
    2. Build a reference index dict.
    3. Annotate every in-text reference with a brief inline description.
    """
    print(f"\n{'─'*60}")
    print(f"  STAGE · REFERENCE RESOLUTION")
    print(f"{'─'*60}")
    print(f"  [IN]  full_markdown length : {len(full_markdown)} chars")
    print(f"  [IN]  figures/tables total : {len(all_figures)}")

    # ── Step 1: Assign global numbers ────────────────────────────────────
    sorted_elems  = sorted(all_figures, key=lambda e: (e["page"], e.get("figure_tag", "")))
    fig_count     = 0
    tbl_count     = 0
    index: dict[str, dict] = {}

    print(f"\n  [OUT] Global numbering :")
    print(f"  {'LOCAL TAG':<12}  {'GLOBAL TAG':<12}  PAGE  CAPTION")
    print(f"  {'─'*12}  {'─'*12}  {'─'*4}  {'─'*35}")

    for elem in sorted_elems:
        if elem.get("type") == "figure":
            fig_count += 1
            gtag = f"Figure {fig_count}"
        else:
            tbl_count += 1
            gtag = f"Table {tbl_count}"

        elem["global_tag"] = gtag
        index[gtag]        = elem
        print(f"  {elem.get('figure_tag','?'):<12}  {gtag:<12}  "
              f"p{elem['page']:<3}  {elem.get('caption','')[:35]}")

        # Sub-panel keys: "Figure 4a", "Figure 4b"
        for sp in elem.get("subplots", []):
            sub_key        = f"{gtag}{sp['label'].lower()}"
            index[sub_key] = {**elem, "subplot_label": sp["label"], "content": sp["content"]}
            print(f"  {'':12}  {sub_key:<12}  (subplot)")

    # ── Step 2: Regex replace in Markdown ────────────────────────────────
    pattern = re.compile(r"\b(fig(?:ure)?|table)\.?\s*(\d+)([a-z]?)\b", re.IGNORECASE)

    match_count = len(pattern.findall(full_markdown))
    print(f"\n  [IN]  Reference matches found in markdown : {match_count}")

    def _annotate(m: re.Match) -> str:
        prefix   = "Figure" if m.group(1).lower().startswith("fig") else "Table"
        key      = f"{prefix} {m.group(2)}{m.group(3).lower()}"
        elem     = index.get(key)
        if not elem:
            return m.group(0)
        snippet  = elem.get("content", "")[:100].rstrip()
        if len(elem.get("content", "")) > 100:
            snippet += "…"
        return f"{m.group(0)} *({snippet} — p.{elem['page']})*"

    enriched = pattern.sub(_annotate, full_markdown)

    replaced = len(pattern.findall(full_markdown)) - len(pattern.findall(enriched.replace("*(", "")))
    print(f"  [OUT] References annotated : {match_count}")
    print(f"  [OUT] Index keys           : {list(index.keys())}")

    return enriched, index

## Process Page

In [33]:
def process_page(page_num: int) -> tuple[str, list[dict]]:
    """
    Full pipeline for a single page.

    INPUT  : page_num (1-based int)
    OUTPUT : (markdown str, figures_tables list[dict])

    Stages
    ------
    1. Render page → BGR image
    2. get_yolo_outputs()    → blocks
    3. get_reading_order()   → reading_order JSON   (LLM Call 1)
    4. generate_markdown()   → markdown + figures   (LLM Call 2)
    """
    print("\n" + "═" * 60)
    print(f"  PROCESS PAGE {page_num}")
    print("═" * 60)

    # Stage 1 — Render
    page     = doc[page_num - 1]
    page_img = render_page(page)
    print(f"\n  [Stage 1] Page rendered  →  shape={page_img.shape}")

    # Stage 2 — YOLO
    blocks = get_yolo_outputs(page_img, page_num)
    if not blocks:
        print(f"  [Stage 2] No blocks — skipping LLM calls.")
        return f"\n\n## Page {page_num}\n\n[No content detected]\n", []

    # Stage 3 — Reading order (LLM Call 1)
    reading_order = get_reading_order(page_img, blocks, page_num)

    # Stage 4 — Markdown + JSON (LLM Call 2)
    markdown, figures_json = generate_markdown(page_img, blocks, reading_order, page_num)

    print(f"\n  {'─'*50}")
    print(f"  PAGE {page_num} DONE  |  "
          f"markdown={len(markdown)} chars  |  elements={len(figures_json)}")
    print(f"  {'─'*50}")

    return f"\n\n## Page {page_num}\n\n{markdown}\n", figures_json

## Run All Pages

In [34]:
# ── Run over every page ──────────────────────────────────────────────────────
all_page_markdowns: dict[int, str]  = {}
all_figures:        list[dict]      = []
total_pages = len(doc)

for page_num in range(1, total_pages + 1):
    try:
        page_md, page_figs = process_page(page_num)
        all_page_markdowns[page_num] = page_md
        all_figures.extend(page_figs)
    except Exception as exc:
        print(f"  [ERROR] page {page_num}: {exc}")
        all_page_markdowns[page_num] = f"\n\n## Page {page_num}\n\n[Failed: {exc}]\n"

print("\n" + "═"*60)
print(f"  ALL PAGES PROCESSED")
print(f"  Pages     : {total_pages}")
print(f"  Elements  : {len(all_figures)}")
print("═"*60)


════════════════════════════════════════════════════════════
  PROCESS PAGE 1
════════════════════════════════════════════════════════════

  [Stage 1] Page rendered  →  shape=(1684, 1191, 3)

────────────────────────────────────────────────────────────
  STAGE 1 · YOLO DETECTION  —  Page 1
────────────────────────────────────────────────────────────
  [IN]  page_img shape : (1684, 1191, 3)  dtype=uint8

0: 1024x736 3 titles, 8 plain texts, 2 abandons, 1 figure, 1 figure_caption, 2812.1ms
Speed: 8.3ms preprocess, 2812.1ms inference, 0.0ms postprocess per image at shape (1, 3, 1024, 736)
  [OUT] 15 block(s) detected  (saved to output_OP_RAG\page_001/)

   IDX  LABEL                  CONF  BBOX (x1,y1,x2,y2)
  ────  ────────────────────  ─────  ────────────────────────────
     1  plain text             0.98  ( 140,1229, 580,1470)
     2  plain text             0.98  ( 172, 480, 548,1172)
     3  plain text             0.98  ( 611,1200,1052,1469)
     4  figure                 0.97  ( 6

## Cell 11 · Resolve References + Save Final Output

In [35]:
# ── Assemble full markdown ────────────────────────────────────────────────────
full_markdown = "".join(all_page_markdowns[p] for p in range(1, total_pages + 1))

# ── Resolve references ────────────────────────────────────────────────────────
enriched_markdown, reference_index = resolve_references(full_markdown, all_figures)

# ── Save final output files ───────────────────────────────────────────────────
out = Path(OUTPUT_DIR)

# Final Markdown
md_path = out / "document.md"
with open(md_path, "w", encoding="utf-8") as f:
    f.write(f"# {Path(PDF_PATH).stem}\n")
    f.write(enriched_markdown)

# Figures + Tables JSON (strip numpy crops before serialising)
json_path = out / "figures_tables.json"
serialisable = [{k: v for k, v in e.items() if k != "crop"} for e in all_figures]
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(serialisable, f, indent=2, ensure_ascii=False)

print(f"\n  [SAVED] {md_path}")
print(f"  [SAVED] {json_path}")
print(f"\n  Output folder structure:")
print(f"  {OUTPUT_DIR}/")
print(f"  ├── document.md          ← final enriched markdown")
print(f"  ├── figures_tables.json  ← all figures & tables structured")
for p in range(1, total_pages + 1):
    print(f"  ├── page_{p:03d}/")
    print(f"  │     ├── block_XX_<label>.jpg   ← YOLO crops")
    print(f"  │     ├── reading_order.json")
    print(f"  │     ├── markdown.md")
    print(f"  │     └── figures.json")


────────────────────────────────────────────────────────────
  STAGE · REFERENCE RESOLUTION
────────────────────────────────────────────────────────────
  [IN]  full_markdown length : 19867 chars
  [IN]  figures/tables total : 5

  [OUT] Global numbering :
  LOCAL TAG     GLOBAL TAG    PAGE  CAPTION
  ────────────  ────────────  ────  ───────────────────────────────────
  Figure 1      Figure 1      p1    Figure 1: Comparisons between the p
                Figure 1a     (subplot)
                Figure 1b     (subplot)
  Figure 1      Figure 2      p2    Figure 2: Vanilla RAG versus the pr
  Figure 1      Figure 3      p3    Figure 3: The influence of context 
                Figure 3a     (subplot)
                Figure 3b     (subplot)
  Figure 1      Figure 4      p4    Figure 4: Comparisons between the p
                Figure 4a     (subplot)
                Figure 4b     (subplot)
  Table 1       Table 1       p4    Table 1: Comparisons among the long

  [IN]  Reference matches